![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Taller: Construcción e implementación de modelos Bagging, Random Forest y XGBoost

En este taller podrán poner en práctica sus conocimientos sobre la construcción e implementación de modelos de Bagging, Random Forest y XGBoost. El taller está constituido por 8 puntos, en los cuales deberan seguir las intrucciones de cada numeral para su desarrollo.

## Datos predicción precio de automóviles

En este taller se usará el conjunto de datos de Car Listings de Kaggle donde cada observación representa el precio de un automóvil teniendo en cuenta distintas variables como año, marca, modelo, entre otras. El objetivo es predecir el precio del automóvil. Para más detalles puede visitar el siguiente enlace: [datos](https://www.kaggle.com/jpayne/852k-used-car-listings).

In [103]:
import warnings
warnings.filterwarnings('ignore')

In [104]:
# Importación de librerías
%matplotlib inline
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

# Lectura de la información de archivo .csv
data = pd.read_csv('https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/datasets/dataTrain_carListings.zip')

# Preprocesamiento de datos para el taller
data = data.loc[data['Model'].str.contains('Camry')].drop(['Make', 'State'], axis=1)
data = data.join(pd.get_dummies(data['Model'], prefix='M'))
data = data.drop(['Model'], axis=1)

# Visualización dataset
data.head()

,Price,Year,Mileage,M_Camry,M_Camry4dr,M_CamryBase,M_CamryL,M_CamryLE,M_CamrySE,M_CamryXLE
7,21995,2014,6480,0,0,0,1,0,0,0
11,13995,2014,39972,0,0,0,0,1,0,0
167,17941,2016,18989,0,0,0,0,0,1,0
225,12493,2014,51330,0,0,0,1,0,0,0
270,7994,2007,116065,0,1,0,0,0,0,0


In [105]:
# Separación de variables predictoras (X) y variable de interés (y)
y = data['Price']
X = data.drop(['Price'], axis=1)

In [106]:
# Separación de datos en set de entrenamiento y test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

### Punto 1 - Árbol de decisión manual

En la celda 1 creen un árbol de decisión **manualmente**  que considere los set de entrenamiento y test definidos anteriormente y presenten el RMSE y MAE del modelo en el set de test.

In [107]:
# Celda 1

# Función MSE como criterio en problemas de regresión para hacer cada split del árbol
def mse(y):
    return np.mean((y - y.mean())**2)

# Función para calcular MSE del split
def mse_split(x, y, split):
    
    left = y[x < split]
    right = y[x >= split]
    
    # Evitar divisiones inválidas
    if len(left) == 0 or len(right) == 0:
        return np.inf
    
    mse_left = mse(left)
    mse_right = mse(right)
    mse_total = (len(left)/len(y))*mse_left + (len(right)/len(y))*mse_right
    
    return mse_total

# Definición de la función best_split para calcular cuál es la mejor variable y punto de corte para hacer la bifurcación del árbol
def best_split(X, y, num_pct=10):
    
    features = range(X.shape[1])
    best_split = [0, 0, np.inf]  # j, split, score
    
    # Para todas las varibles 
    for j in features:
        
        splits = np.percentile(X.iloc[:, j], np.arange(0, 100, 100.0 / (num_pct+1)).tolist())
        splits = np.unique(splits)[1:]
        
        # Para cada partición
        for split in splits:
            score = mse_split(X.iloc[:, j], y, split)
                        
            if score  < best_split[2]:
                best_split = [j, split, score]
    
    return best_split

# Definición de la función tree_grow para hacer un crecimiento recursivo del árbol de regresión
def tree_grow(X, y, level=0, min_gain=1000, max_depth=None, num_pct=10):
    
    # Si solo es una observación
    if X.shape[0] == 1:
        tree = dict(y_pred=y.iloc[:1].values[0], level=level, split=-1, n_samples=1, score=0)
        return tree
    
    # Calcular la mejor división
    j, split, score = best_split(X, y, num_pct)
    
    # Predicción = media
    y_pred = y.mean()
    tree = dict(y_pred=y_pred, level=level, split=-1, n_samples=X.shape[0], score=score)
    
    # Calcular ganancia
    gain = mse(y) - score

    # Revisar el criterio de parada 
    if gain < min_gain:
        return tree
    if max_depth is not None:
        if level >= max_depth:
            return tree   
     
    # Continuar creando la partición
    filter_l = X.iloc[:, j] < split
    X_l, y_l = X.loc[filter_l], y.loc[filter_l]
    X_r, y_r = X.loc[~filter_l], y.loc[~filter_l]
    tree['split'] = [j, split]

    # Siguiente iteración para cada partición
    tree['sl'] = tree_grow(X_l, y_l, level + 1, min_gain=min_gain, max_depth=max_depth, num_pct=num_pct)
    tree['sr'] = tree_grow(X_r, y_r, level + 1, min_gain=min_gain, max_depth=max_depth, num_pct=num_pct)
    
    return tree

# Definición de la función tree_predict para hacer predicciones según las variables 'X' y el árbol 'tree'
def tree_predict(X, tree):
    
    predicted = np.ones(X.shape[0])

    # Revisar si es el nodo final
    if tree['split'] == -1:
        predicted = predicted * tree['y_pred']
            
    else:
        
        j, split = tree['split']
        filter_l = (X.iloc[:, j] < split)
        X_l = X.loc[filter_l]
        X_r = X.loc[~filter_l]

        if X_l.shape[0] == 0:  # Si el nodo izquierdo está vacio solo continua con el derecho 
            predicted[~filter_l] = tree_predict(X_r, tree['sr'])
        elif X_r.shape[0] == 0:  #  Si el nodo derecho está vacio solo continua con el izquierdo
            predicted[filter_l] = tree_predict(X_l, tree['sl'])
        else:
            predicted[filter_l] = tree_predict(X_l, tree['sl'])
            predicted[~filter_l] = tree_predict(X_r, tree['sr'])

    return predicted


In [108]:
#Entrenar arbol usando train
tree = tree_grow(X_train, y_train, level=0, min_gain=1000, max_depth=5, num_pct=10)
tree

# Ejecución de función tree_predict usando test
y_pred_test = tree_predict(X_test, tree)
print("Valores predichos:")
print(y_pred_test)

#Cálculo del RMSE y MAE 
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae = mean_absolute_error(y_test, y_pred_test)
print("\nRMSE Árbol manual:", rmse)
print("MAE Árbol manual:", mae)


Valores predichos:
[13194.57990868  7785.453125   16170.66612378 ... 18100.90422535
 12503.56140351 10789.80327869]

RMSE Árbol manual: 1697.9641986149657
MAE Árbol manual: 1267.7909847691142


### Punto 2 - Bagging manual

En la celda 2 creen un modelo bagging **manualmente** con 10 árboles de regresión y comenten sobre el desempeño del modelo.

In [140]:
# Celda 2

#Resetear los indices antes de hacer el bagging para poder realizar las muestras de bootstrap
X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

# Se crea un arreglo de 1 a 7100 y se imprime el arreglo y muestreo aleatorio
np.random.seed(123)
nums = np.arange(0, 7100)
print('Arreglo:', nums)
print('Muestreo aleatorio: ', np.random.choice(a=nums, size=7100, replace=True))

# Creación de 10 muestras de bootstrap 
n_samples = X_train.shape[0]
n_B = 10
samples = [np.random.choice(a=n_samples, size=n_samples, replace=True) for _ in range(1, n_B +1 )]
samples

# Visualización muestra boostrap #1 para entremiento
print('\nMuestra Boostrap #1:')
X_train.iloc[samples[0], :]

Arreglo: [   0    1    2 ... 7097 7098 7099]
Muestreo aleatorio:  [3582 3454 1346 ... 5863 3492 4254]

Muestra Boostrap #1:


,Year,Mileage,M_Camry,M_Camry4dr,M_CamryBase,M_CamryL,M_CamryLE,M_CamrySE,M_CamryXLE
4964,2012,83553,0,1,0,0,0,0,0
2938,2017,30929,0,0,0,0,0,1,0
3462,2014,44020,0,0,0,1,0,0,0
3447,2017,2008,0,0,0,0,0,1,0
4777,2007,117433,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
4391,2006,124735,0,1,0,0,0,0,0
484,2014,57487,0,0,0,0,1,0,0
2521,2011,89766,0,1,0,0,0,0,0
2466,2012,57859,0,0,0,0,0,1,0


In [141]:
# Definición del modelo usando DecisionTreeRegressor de sklearn
treereg = DecisionTreeRegressor(max_depth=None, random_state=123) 

# DataFrame para guardar las predicciones de cada árbol
y_pred = pd.DataFrame(index=X_test.index, columns=[list(range(n_B))])

# Entrenamiento de un árbol sobre cada muestra boostrap y predicción sobre los datos de test
for i, sample in enumerate(samples):
    X_boot  = X_train.iloc[sample, :]
    y_boot  = y_train.iloc[sample]
    treereg.fit(X_boot, y_boot)
    y_pred.iloc[:,i] = treereg.predict(X_test)
    
y_pred

,0,1,2,3,4,5,6,7,8,9
257343,13993.0,13649.0,13649.0,13990.0,13649.0,13993.0,13993.0,13990.0,13993.0,13649.0
326011,5995.0,5995.0,6987.0,5995.0,5995.0,5995.0,6987.0,5995.0,5995.0,5995.0
242354,16995.0,16491.0,15997.0,15997.0,16491.0,17591.0,16995.0,17404.0,16491.0,16491.0
266376,21990.0,22500.0,21990.0,15900.0,21990.0,21990.0,21990.0,15813.0,21990.0,15900.0
396954,16951.0,15988.0,15988.0,15988.0,17900.0,16951.0,16951.0,15988.0,15988.0,15988.0
...,...,...,...,...,...,...,...,...,...,...
144298,14800.0,14800.0,14800.0,14800.0,14681.0,14800.0,14800.0,14800.0,13836.0,14800.0
364521,14995.0,15999.0,17987.0,15999.0,15999.0,15999.0,15999.0,16999.0,14851.0,15999.0
120072,23533.0,20000.0,17700.0,17700.0,23533.0,17700.0,23533.0,23533.0,20000.0,17700.0
99878,12991.0,12989.0,12995.0,12991.0,12991.0,10995.0,12991.0,12991.0,12893.0,12989.0


In [148]:
# Desempeño de cada árbol
for i in range(n_B):
    print('Árbol ', i, 'tiene un RMSE: ', np.sqrt(mean_squared_error(y_test,y_pred.iloc[:,i])))


# Predicciones promedio para cada observación del set de test
y_pred.mean(axis=1)

#Cálculo del RMSE y MAE 
np.sqrt(mean_squared_error(y_test, y_pred.mean(axis=1)))

#Cálculo del RMSE y MAE al promediar las predicciones de todos los árboles
rmse = np.sqrt(mean_squared_error(y_test, y_pred.mean(axis=1)))
mae = mean_absolute_error(y_test, y_pred.mean(axis=1))
print("\nRMSE Bagging manual:", rmse)
print("MAE Bagging manual:", mae)

Árbol  0 tiene un RMSE:  2141.9171501683118
Árbol  1 tiene un RMSE:  2125.6987529878875
Árbol  2 tiene un RMSE:  2082.9814991396815
Árbol  3 tiene un RMSE:  2165.747795521846
Árbol  4 tiene un RMSE:  2116.26408903415
Árbol  5 tiene un RMSE:  2153.0973164388097
Árbol  6 tiene un RMSE:  2194.303636378142
Árbol  7 tiene un RMSE:  2142.4593794405823
Árbol  8 tiene un RMSE:  2136.6026071273845
Árbol  9 tiene un RMSE:  2135.7804645213514

RMSE Bagging manual: 1798.8880759942176
MAE Bagging manual: 1341.2787475943035


**Interpretación**

El desempeño de los árboles individuales muestra errores relativamente similares, con valores de RMSE entre aproximadamente 2083 y 2194, lo que indica que cada árbol tiene una capacidad predictiva parecida, aunque con cierta variabilidad debido al muestreo bootstrap. Sin embargo, al aplicar bagging y promediar las predicciones de los 10 árboles, el error disminuye de forma notable, obteniendo un RMSE de 1798.89 y un MAE de 1341.28. Este resultado es considerablemente mejor que el de cualquiera de los árboles individuales, lo que evidencia que el bagging logra reducir la varianza del modelo, ya que la combinación de múltiples árboles mediante promediado permite obtener predicciones más robustas y precisas.

### Punto 3 - Bagging con librería

En la celda 3, con la librería sklearn, entrenen un modelo bagging con 10 árboles de regresión y el parámetro `max_features` del árbol de decisión igual a `log(n_features)` y comenten sobre el desempeño del modelo.

In [113]:
# Celda 3


### Punto 4 - Random forest con librería

En la celda 4, usando la librería sklearn entrenen un modelo de Randon Forest para regresión  y comenten sobre el desempeño del modelo.

In [114]:
# Celda 4


### Punto 5 - Calibración de parámetros Random forest

En la celda 5, calibren los parámetros max_depth, max_features y n_estimators del modelo de Randon Forest para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [115]:
# Celda 5


### Punto 6 - XGBoost con librería

En la celda 6 implementen un modelo XGBoost de regresión con la librería sklearn y comenten sobre el desempeño del modelo.

In [116]:
# Celda 6


### Punto 7 - Calibración de parámetros XGBoost

En la celda 7 calibren los parámetros learning rate, gamma y colsample_bytree del modelo XGBoost para regresión, comenten sobre el desempeño del modelo y describan cómo cada parámetro afecta el desempeño del modelo.

In [117]:
# Celda 7


### Punto 8 - Comparación y análisis de resultados
En la celda 8 comparen los resultados obtenidos de los diferentes modelos (random forest y XGBoost) y comenten las ventajas del mejor modelo y las desventajas del modelo con el menor desempeño.

In [118]:
# Celda 8
